# YOLO — Modèle BÉNÉFICIAIRE
**Classe unique :** `beneficiaire` (class 0)

**Pipeline :** annotez avec MakeSense.ai → exportez ZIP YOLO → uploadez sur Drive → lancez ce notebook.

In [ ]:
# ── ÉTAPE 1 : Installation ──────────────────────────────────────────
!pip install ultralytics --quiet

In [ ]:
# ── ÉTAPE 2 : Monter Google Drive ───────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ── ÉTAPE 3 : Configuration — MODIFIEZ CES CHEMINS ──────────────────
import os

# Chemin du ZIP exporté depuis MakeSense.ai (sur votre Drive)
ZIP_PATH   = '/content/drive/MyDrive/tunicheque/dataset_benef.zip'

# Dossier de sortie du dataset
DATASET_DIR = '/content/dataset_benef'

# Dossier où sauvegarder best.pt après entraînement
SAVE_DIR    = '/content/drive/MyDrive/tunicheque/models/yolo_benef'

os.makedirs(DATASET_DIR, exist_ok=True)
os.makedirs(SAVE_DIR,    exist_ok=True)
print('OK — chemins configurés')

In [ ]:
# ── ÉTAPE 4 : Décompression du dataset ──────────────────────────────
import zipfile, shutil, pathlib

with zipfile.ZipFile(ZIP_PATH, 'r') as z:
    z.extractall(DATASET_DIR)

# Afficher la structure extraite
for p in sorted(pathlib.Path(DATASET_DIR).rglob('*'))[:30]:
    print(p)

In [ ]:
# ── ÉTAPE 5 : Organiser train/val (80/20) ───────────────────────────
import pathlib, shutil, random

# MakeSense.ai exporte : images/ + labels/
# Adaptez selon la structure réelle de votre ZIP
src = pathlib.Path(DATASET_DIR)

# Cherche les images (jpg/png) et leurs labels (.txt)
images = sorted(src.rglob('*.jpg')) + sorted(src.rglob('*.png'))
print(f'{len(images)} images trouvées')

random.seed(42)
random.shuffle(images)
split = int(len(images) * 0.8)
train_imgs, val_imgs = images[:split], images[split:]

for split_name, split_list in [('train', train_imgs), ('val', val_imgs)]:
    img_dir = src / split_name / 'images'
    lbl_dir = src / split_name / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    for img in split_list:
        shutil.copy(img, img_dir / img.name)
        lbl = img.with_suffix('.txt')
        if lbl.exists():
            shutil.copy(lbl, lbl_dir / lbl.name)

print(f'Train : {len(train_imgs)} | Val : {len(val_imgs)}')

In [ ]:
# ── ÉTAPE 6 : Créer le fichier YAML ─────────────────────────────────
yaml_content = f"""path: {DATASET_DIR}
train: train/images
val:   val/images

nc: 1
names:
  0: beneficiaire
"""

yaml_path = '/content/benef.yaml'
with open(yaml_path, 'w') as f:
    f.write(yaml_content)

print(yaml_content)

In [ ]:
# ── ÉTAPE 7 : Entraînement YOLOv8 ───────────────────────────────────
from ultralytics import YOLO

model = YOLO('yolov8n.pt')   # nano = rapide sur Colab gratuit

results = model.train(
    data    = yaml_path,
    epochs  = 100,
    imgsz   = 640,
    batch   = 8,
    patience= 20,           # early stopping si pas d'amélioration
    name    = 'yolo_benef',
    project = '/content/runs',
    exist_ok= True,
    device  = 0,            # GPU Colab
    augment = True,
    degrees = 5,            # légère rotation
    flipud  = 0.0,          # ne pas retourner les chèques
    fliplr  = 0.3,
)

print('Entraînement terminé !')

In [ ]:
# ── ÉTAPE 8 : Copier best.pt sur Drive ──────────────────────────────
import shutil, pathlib

best = pathlib.Path('/content/runs/yolo_benef/weights/best.pt')
dest = pathlib.Path(SAVE_DIR) / 'best.pt'

shutil.copy(best, dest)
print(f'best.pt sauvegardé → {dest}')

In [ ]:
# ── ÉTAPE 9 : Test rapide sur une image ─────────────────────────────
from ultralytics import YOLO
import pathlib

model = YOLO(str(pathlib.Path(SAVE_DIR) / 'best.pt'))

# Remplacez par une image de test
TEST_IMAGE = '/content/drive/MyDrive/tunicheque/test_cheque.jpg'

results = model(TEST_IMAGE, conf=0.25, verbose=True)
for r in results:
    print(f'{len(r.boxes)} détection(s)')
    for b in r.boxes:
        print(f'  conf={float(b.conf[0]):.3f}  bbox={b.xyxy[0].tolist()}')